<a href="https://colab.research.google.com/github/nkhrb/Discharge-prediction-of-mahanadi-river-basin-using-machine-learning-models/blob/main/main_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# COMPLETE RIVER DISCHARGE FORECASTING SCRIPT FOR GOOGLE COLAB
# Expert Data Scientist and Hydrologist - River Flow Forecasting
# v2: Rating Curve (COM1 only) + Best-COM summary table (RMSE/MAE/R²)
# =====================================================================

# =====================================================================
# PART 1: IMPORTS AND SETUP
# =====================================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime, timedelta
from scipy.optimize import curve_fit
from scipy.stats import pearsonr
warnings.filterwarnings('ignore')

# Machine Learning Libraries
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR, LinearSVR

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, Conv1D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Wavelet Transform
import pywt

print("✓ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"PyWavelets version: {pywt.__version__}\n")

# =====================================================================
# PART 2: COM FEATURE COMBINATIONS
# =====================================================================
COM_FEATURES = {
    'COM1': ['S_t'],
    'COM2': ['S_t', 'S_lag1'],
    'COM3': ['S_t', 'S_lag1', 'S_lag2'],
    'COM4': ['S_t', 'S_lag1', 'S_lag2', 'S_lag3'],
    'COM5': ['S_t', 'S_lag1', 'S_lag2', 'S_lag3', 'Q_lag1'],
    'COM6': ['S_t', 'S_lag1', 'S_lag2', 'S_lag3', 'Q_lag1', 'Q_lag2'],
    'COM7': ['S_t', 'S_lag1', 'S_lag2', 'S_lag3', 'Q_lag1', 'Q_lag2', 'Q_lag3']
}

def print_com_features_table():
    """Prints a formatted table of all COM feature combinations."""
    print("\n" + "="*70)
    print("COM FEATURE COMBINATIONS")
    print("="*70)
    print(f"{'Combination':<12} {'No. Features':<15} {'Features'}")
    print("-"*70)
    for com_name, features in COM_FEATURES.items():
        print(f"{com_name:<12} {len(features):<15} {', '.join(features)}")
    print("-"*70)
    print("  S_t     = Current day stage (water level)")
    print("  S_lag1  = Stage 1 day ago")
    print("  S_lag2  = Stage 2 days ago")
    print("  S_lag3  = Stage 3 days ago")
    print("  Q_lag1  = Discharge 1 day ago")
    print("  Q_lag2  = Discharge 2 days ago")
    print("  Q_lag3  = Discharge 3 days ago")
    print("  NOTE: Rating Curve is a statistical method; it uses COM1 (S_t) only.")
    print("="*70)

# =====================================================================
# PART 3: DATA PREPROCESSING FUNCTION (MULTI-BLOCK YEAR DETECTION)
# =====================================================================
def preprocess_data(filepath, start_year_override=None):
    """
    Preprocesses discharge data from non-standard wide format Excel files.
    Returns DataFrame with columns [Date, Stage, Discharge] sorted by date.
    """
    print(f"\n{'='*70}")
    print(f"Processing: {filepath.split('/')[-1]}")
    print(f"{'='*70}")

    df = pd.read_excel(filepath, header=None, dtype=object)

    def extract_year_from_text(text):
        if text is None:
            return None
        text = str(text)
        m = re.search(r'(?:19|20)\d{2}', text)
        return int(m.group(0)) if m else None

    year_cells = []
    year_pattern = re.compile(r'(?:19|20)\d{2}(?:\s*[-/]\s*(?:19|20)\d{2})?')
    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            val = df.iat[r, c]
            if pd.isna(val):
                continue
            try:
                if year_pattern.search(str(val)):
                    year_cells.append((r, c, str(val)))
            except Exception:
                continue

    year_rows = sorted({(r, c, txt) for (r, c, txt) in year_cells}, key=lambda x: (x[0], x[1]))

    if start_year_override is not None:
        print(f"Using start_year_override = {start_year_override} for file {filepath}")

    month_map = {
        'jun': 6, 'june': 6,
        'jul': 7, 'july': 7,
        'aug': 8, 'august': 8,
        'sep': 9, 'sept': 9, 'september': 9,
        'oct': 10, 'october': 10,
        'nov': 11, 'november': 11,
        'dec': 12, 'december': 12,
        'jan': 1, 'january': 1,
        'feb': 2, 'february': 2,
        'mar': 3, 'march': 3,
        'apr': 4, 'april': 4,
        'may': 5
    }

    records = []

    if not year_rows:
        print("No year header cells found scanning the sheet. Trying fallback detection.")
        start_year = None
        try:
            year_cell = df.iat[0, 0]
            start_year = extract_year_from_text(year_cell)
            if start_year:
                print(f"Detected year in top-left: {year_cell} -> {start_year}")
        except Exception:
            start_year = None

        if start_year is None:
            header_text = ""
            for i in range(min(3, df.shape[0])):
                header_text += " " + " ".join([str(x) for x in df.iloc[i].values if not pd.isna(x)])
            start_year = extract_year_from_text(header_text)
            if start_year:
                print(f"Detected year in header rows -> {start_year}")

        if start_year is None:
            start_year = extract_year_from_text(filepath)
            if start_year:
                print(f"Detected year in filename -> {start_year}")

        if start_year is None and start_year_override is None:
            raise ValueError(f"Could not detect any year header in '{filepath}'.")
        block_rows = [(0, 0, str(df.iat[0, 0]) if df.shape[0] > 0 and not pd.isna(df.iat[0, 0]) else '')]
    else:
        block_rows = year_rows

    used_row_ranges = []
    for (r, c, txt) in block_rows:
        start_year_block = None
        if start_year_override is not None:
            start_year_block = int(start_year_override)
        else:
            start_year_block = extract_year_from_text(txt)

        if start_year_block is None:
            continue

        months_row_idx = r + 1
        labels_row_idx = r + 2
        data_start_idx = r + 3

        if months_row_idx >= df.shape[0] or labels_row_idx >= df.shape[0]:
            continue

        months_row = df.iloc[months_row_idx, c+1:df.shape[1]].reset_index(drop=True)
        labels_row = df.iloc[labels_row_idx, c+1:df.shape[1]].reset_index(drop=True)

        local_col = c + 1
        while local_col < df.shape[1]:
            idx_in_months = local_col - (c + 1)
            if idx_in_months >= len(months_row):
                break
            month_name = months_row.iloc[idx_in_months]
            if pd.isna(month_name):
                local_col += 1
                continue
            mn = str(month_name).strip().lower()
            mn_token = mn.split()[0] if mn else mn
            if mn_token in month_map:
                month_num = month_map[mn_token]
                year_for_month = start_year_block if month_num >= 6 else start_year_block + 1

                q_col = local_col
                wl_col = None
                for offset in range(1, 5):
                    lbl_idx = (local_col - (c + 1)) + offset
                    if lbl_idx < len(labels_row):
                        label = labels_row.iloc[lbl_idx]
                        if pd.notna(label) and ('W.L' in str(label) or 'WL' in str(label)
                                                or 'W L' in str(label) or 'Water' in str(label)):
                            wl_col = c + 1 + lbl_idx
                            break
                if wl_col is None:
                    for attempt in range(1, 3):
                        cand = q_col + attempt
                        lbl_idx = cand - (c + 1)
                        if 0 <= lbl_idx < len(labels_row):
                            if pd.notna(labels_row.iloc[lbl_idx]):
                                if 'Q' not in str(labels_row.iloc[lbl_idx]):
                                    wl_col = cand
                                    break
                            else:
                                wl_col = cand
                                break

                if wl_col is not None and q_col < df.shape[1] and wl_col < df.shape[1]:
                    for day_idx in range(data_start_idx, df.shape[0]):
                        day_val = df.iat[day_idx, 0] if 0 < df.shape[1] else None
                        if pd.isna(day_val):
                            continue
                        try:
                            day_num = int(float(day_val))
                        except Exception:
                            break

                        try:
                            q_val = df.iat[day_idx, q_col]
                        except Exception:
                            q_val = np.nan
                        try:
                            wl_val = df.iat[day_idx, wl_col]
                        except Exception:
                            wl_val = np.nan

                        if pd.isna(q_val) or pd.isna(wl_val):
                            continue

                        q_str = str(q_val).strip()
                        wl_str = str(wl_val).strip()
                        if '*' in q_str or '*' in wl_str or q_str == '0' or wl_str == '0':
                            continue

                        try:
                            q_float = float(q_str)
                            wl_float = float(wl_str)
                        except Exception:
                            continue

                        if q_float == 0 or wl_float == 0:
                            continue

                        try:
                            date = datetime(year_for_month, month_num, day_num)
                        except ValueError:
                            continue

                        records.append({'Date': date, 'Stage': wl_float, 'Discharge': q_float})
                local_col += 1
            else:
                local_col += 1

        used_row_ranges.append((r, months_row_idx, labels_row_idx))

    result_df = pd.DataFrame(records)

    if result_df.empty:
        print("⚠ WARNING: No valid records extracted from any detected blocks!")
        return pd.DataFrame(columns=['Date', 'Stage', 'Discharge'])

    result_df = result_df.drop_duplicates(subset=['Date', 'Stage', 'Discharge'])
    result_df = result_df.sort_values('Date').reset_index(drop=True)

    print(f"\nExtracted Records: {len(result_df)}")
    print(f"Date Range: {result_df['Date'].min()} to {result_df['Date'].max()}")
    print(f"\nDischarge Statistics:")
    print(result_df['Discharge'].describe())
    print(f"\nStage Statistics:")
    print(result_df['Stage'].describe())

    return result_df

# =====================================================================
# PART 4: FEATURE ENGINEERING WITH COM COMBINATIONS
# =====================================================================
def create_com_features(df, com_name, feature_list):
    """
    Creates feature matrix for a given COM combination.
    Target is Q_t (current discharge).
    Returns: X, y, dates
    """
    df = df.copy()
    df['S_t']    = df['Stage']
    df['S_lag1'] = df['Stage'].shift(1)
    df['S_lag2'] = df['Stage'].shift(2)
    df['S_lag3'] = df['Stage'].shift(3)
    df['Q_lag1'] = df['Discharge'].shift(1)
    df['Q_lag2'] = df['Discharge'].shift(2)
    df['Q_lag3'] = df['Discharge'].shift(3)

    needed_cols = feature_list + ['Discharge', 'Date']
    df = df[needed_cols].dropna().reset_index(drop=True)

    X = df[feature_list].values
    y = df['Discharge'].values
    dates = df['Date'].values
    return X, y, dates


def create_features(df):
    """Legacy wrapper kept for backward compatibility."""
    return create_com_features(df, 'legacy', ['S_t', 'S_lag1', 'Q_lag1'])

# =====================================================================
# PART 5: WAVELET DECOMPOSITION FUNCTION
# =====================================================================
def wavelet_decomposition(signal, wavelet='db4', level=3):
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    coeffs_denoised = [coeffs[0]] + [np.zeros_like(c) for c in coeffs[1:]]
    signal_denoised = pywt.waverec(coeffs_denoised, wavelet)
    if len(signal_denoised) > len(signal):
        signal_denoised = signal_denoised[:len(signal)]
    elif len(signal_denoised) < len(signal):
        signal_denoised = np.pad(signal_denoised, (0, len(signal) - len(signal_denoised)), 'edge')
    return signal_denoised

# =====================================================================
# PART 6: RATING CURVE — STATISTICAL METHOD (COM1 ONLY)
# =====================================================================
def train_rating_curve(stage_train, q_train, stage_test):
    """
    Fits a power-law rating curve  Q = a * (H - b)^c  to training data,
    then predicts discharge on the test stage values.

    The gauge datum offset b is estimated as (min stage - small epsilon)
    so that (H - b) > 0 for all observations.

    Falls back to a simple power law Q = a * H^c if the offset fit fails.

    Args:
        stage_train : 1-D array of training stage values (H)
        q_train     : 1-D array of training discharge values (Q)
        stage_test  : 1-D array of test stage values

    Returns:
        y_pred      : predicted discharge on test set
        params      : dict with fitted parameters and equation string
    """
    print("  [RC] Fitting Rating Curve (power-law: Q = a*(H-b)^c) ...")

    # Estimate datum offset b
    b_est = max(0.0, stage_train.min() - 0.01)

    def power_law_offset(H, a, c):
        return a * np.power(np.maximum(H - b_est, 1e-6), c)

    def power_law_simple(H, a, c):
        return a * np.power(np.maximum(H, 1e-6), c)

    fitted_params = None
    equation_str  = ""

    # ── attempt 1: log-space OLS (most robust for power law) ──────────
    try:
        H_eff = stage_train - b_est
        valid = (H_eff > 0) & (q_train > 0)
        if valid.sum() >= 3:
            log_H = np.log(H_eff[valid])
            log_Q = np.log(q_train[valid])
            c_ols, log_a_ols = np.polyfit(log_H, log_Q, 1)
            a_ols = np.exp(log_a_ols)
            fitted_params = (a_ols, c_ols)
            equation_str  = f"Q = {a_ols:.4f} * (H - {b_est:.4f})^{c_ols:.4f}"
            mode = "offset"
    except Exception:
        pass

    # ── attempt 2: scipy curve_fit with offset ─────────────────────────
    if fitted_params is None:
        try:
            popt, _ = curve_fit(power_law_offset, stage_train, q_train,
                                p0=[1.0, 1.5], maxfev=10000,
                                bounds=([0, 0], [np.inf, np.inf]))
            fitted_params = tuple(popt)
            equation_str  = f"Q = {popt[0]:.4f} * (H - {b_est:.4f})^{popt[1]:.4f}"
            mode = "offset"
        except Exception:
            pass

    # ── attempt 3: simple power law (no offset) ────────────────────────
    if fitted_params is None:
        try:
            popt, _ = curve_fit(power_law_simple, stage_train, q_train,
                                p0=[1.0, 1.5], maxfev=10000,
                                bounds=([0, 0], [np.inf, np.inf]))
            fitted_params = tuple(popt)
            b_est  = 0.0
            equation_str  = f"Q = {popt[0]:.4f} * H^{popt[1]:.4f}"
            mode = "simple"
        except Exception as e:
            raise RuntimeError(f"Rating curve fitting failed entirely: {e}")

    a, c = fitted_params

    # Predict
    if mode == "offset":
        y_pred = a * np.power(np.maximum(stage_test - b_est, 1e-6), c)
    else:
        y_pred = a * np.power(np.maximum(stage_test, 1e-6), c)

    params = {'a': a, 'b': b_est, 'c': c,
              'equation': equation_str, 'mode': mode}

    print(f"  ✓ Rating Curve fitted  →  {equation_str}")
    return y_pred, params


def visualize_rating_curve(stage_train, q_train, stage_test, q_test,
                           y_pred_rc, rc_params, station_name, dates_test=None):
    """
    Produces four panels for the rating curve:
      1. Rating curve fit (H vs Q scatter + fitted curve)
      2. Time-series observed vs predicted
      3. Scatter plot observed vs predicted
      4. Residual violin plot
    """
    fig, axes = plt.subplots(1, 4, figsize=(24, 5))
    fig.suptitle(f"{station_name} | Rating Curve | {rc_params['equation']}",
                 fontsize=13, fontweight='bold')

    # ── Panel 1: Rating curve fit ──────────────────────────────────────
    ax = axes[0]
    ax.scatter(stage_train, q_train, alpha=0.5, s=30, color='steelblue',
               edgecolors='k', linewidth=0.3, label='Training data')
    ax.scatter(stage_test,  q_test,  alpha=0.7, s=40, color='orange',
               edgecolors='k', linewidth=0.3, label='Test data (observed)')
    h_range = np.linspace(min(stage_train.min(), stage_test.min()),
                          max(stage_train.max(), stage_test.max()), 300)
    b = rc_params['b']
    a = rc_params['a']
    c = rc_params['c']
    if rc_params['mode'] == 'offset':
        q_fit = a * np.power(np.maximum(h_range - b, 1e-6), c)
    else:
        q_fit = a * np.power(np.maximum(h_range, 1e-6), c)
    ax.plot(h_range, q_fit, 'r-', linewidth=2, label='Fitted curve')
    ax.set_xlabel('Stage / Water Level (m)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Discharge (cumecs)',       fontsize=11, fontweight='bold')
    ax.set_title('Rating Curve Fit',          fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Panel 2: Time series ──────────────────────────────────────────
    ax = axes[1]
    if dates_test is not None:
        ax.plot(dates_test, q_test,   'b-',  label='Observed',  linewidth=2, alpha=0.7)
        ax.plot(dates_test, y_pred_rc,'r--', label='Predicted', linewidth=2, alpha=0.7)
    else:
        ax.plot(q_test,    'b-',  label='Observed',  linewidth=2, alpha=0.7)
        ax.plot(y_pred_rc, 'r--', label='Predicted', linewidth=2, alpha=0.7)
    ax.set_xlabel('Time',                   fontsize=11, fontweight='bold')
    ax.set_ylabel('Discharge (cumecs)',      fontsize=11, fontweight='bold')
    ax.set_title('Time Series',             fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Panel 3: Scatter observed vs predicted ────────────────────────
    ax = axes[2]
    ax.scatter(q_test, y_pred_rc, alpha=0.6, s=50,
               edgecolors='k', linewidth=0.5, color='steelblue')
    lim_min = min(q_test.min(), y_pred_rc.min())
    lim_max = max(q_test.max(), y_pred_rc.max())
    ax.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', linewidth=2, label='1:1 Line')
    r2 = r2_score(q_test, y_pred_rc)
    ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_xlabel('Observed Discharge (cumecs)',  fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted Discharge (cumecs)', fontsize=11, fontweight='bold')
    ax.set_title('Scatter Plot',                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # ── Panel 4: Residual violin ──────────────────────────────────────
    ax = axes[3]
    residuals = q_test - y_pred_rc
    parts = ax.violinplot([residuals], positions=[0], showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor('skyblue')
        pc.set_alpha(0.7)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax.set_ylabel('Prediction Error (Residuals)', fontsize=11, fontweight='bold')
    ax.set_title('Error Distribution',            fontsize=12, fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels(['Rating Curve'])
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

# =====================================================================
# PART 7: MODEL TRAINING FUNCTIONS (ML)
# =====================================================================

def train_random_forest(X_train, y_train, X_test, y_test):
    print("  [1/7] Training Random Forest Regressor...")
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("  ✓ Random Forest complete")
    return y_pred, model

def train_svr(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [2/7] Training SVR (RBF) ...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
    model = SVR(kernel='rbf', C=1.0, epsilon=0.1)
    model.fit(X_train_scaled, y_train_scaled)
    y_pred_scaled = model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ SVR complete")
    return y_pred, model

def train_linear_svr(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [3/7] Training Linear SVM (LinearSVR) ...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
    model = LinearSVR(random_state=42, max_iter=20000)
    model.fit(X_train_scaled, y_train_scaled)
    y_pred_scaled = model.predict(X_test_scaled)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ Linear SVM complete")
    return y_pred, model

def train_lstm(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [4/7] Training LSTM Network...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
    X_test_lstm  = X_test_scaled.reshape(X_test_scaled.shape[0],  1, X_test_scaled.shape[1])
    model = Sequential([
        LSTM(64, activation='relu', return_sequences=True, input_shape=(1, X_train.shape[1])),
        Dropout(0.2),
        LSTM(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(X_train_lstm, y_train_scaled, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    y_pred_scaled = model.predict(X_test_lstm, verbose=0).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ LSTM complete")
    return y_pred, model

def train_gru(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [5/7] Training GRU Network...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    X_train_gru = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
    X_test_gru  = X_test_scaled.reshape(X_test_scaled.shape[0],  1, X_test_scaled.shape[1])
    model = Sequential([
        GRU(64, activation='relu', return_sequences=True, input_shape=(1, X_train.shape[1])),
        Dropout(0.2),
        GRU(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(X_train_gru, y_train_scaled, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    y_pred_scaled = model.predict(X_test_gru, verbose=0).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ GRU complete")
    return y_pred, model

def train_cnn(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [6/7] Training 1D CNN...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    X_train_cnn = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
    X_test_cnn  = X_test_scaled.reshape(X_test_scaled.shape[0],  X_test_scaled.shape[1],  1)
    kernel_size = min(2, X_train.shape[1])
    model = Sequential([
        Conv1D(filters=64, kernel_size=kernel_size, activation='relu',
               input_shape=(X_train.shape[1], 1)),
        Dropout(0.2),
        Conv1D(filters=32, kernel_size=kernel_size, activation='relu'),
        Dropout(0.2),
        Flatten(),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(X_train_cnn, y_train_scaled, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    y_pred_scaled = model.predict(X_test_cnn, verbose=0).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ CNN complete")
    return y_pred, model

def train_ann(X_train, y_train, X_test, y_test, scaler_X, scaler_y):
    print("  [7/7] Training ANN (Multi-Layer Perceptron)...")
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled  = scaler_X.transform(X_test)
    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(X_train_scaled, y_train_scaled, epochs=100, batch_size=32, verbose=0, callbacks=[early_stop])
    y_pred_scaled = model.predict(X_test_scaled, verbose=0).flatten()
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    print("  ✓ ANN complete")
    return y_pred, model

# =====================================================================
# PART 8: EVALUATION METRICS
# =====================================================================
def calculate_metrics(y_true, y_pred, model_name):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    nse  = 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))
    try:
        correlation = np.corrcoef(y_true, y_pred)[0, 1]
    except Exception:
        correlation = np.nan
    return {
        'Model':       model_name,
        'RMSE':        round(rmse, 4),
        'MAE':         round(mae,  4),
        'R²':          round(r2,   4),
        'NSE':         round(nse,  4),
        'Correlation': round(correlation, 4)
    }

# =====================================================================
# PART 9: VISUALIZATION FUNCTIONS (ML models)
# =====================================================================
def visualize_results(y_true, y_pred, model_name, dates=None):
    fig = plt.figure(figsize=(18, 5))

    ax1 = plt.subplot(1, 3, 1)
    if dates is not None:
        ax1.plot(dates, y_true, 'b-',  label='Observed',  linewidth=2, alpha=0.7)
        ax1.plot(dates, y_pred, 'r--', label='Predicted', linewidth=2, alpha=0.7)
    else:
        ax1.plot(y_true, 'b-',  label='Observed',  linewidth=2, alpha=0.7)
        ax1.plot(y_pred, 'r--', label='Predicted', linewidth=2, alpha=0.7)
    ax1.set_xlabel('Time',                  fontsize=12, fontweight='bold')
    ax1.set_ylabel('Discharge (cumecs)',     fontsize=12, fontweight='bold')
    ax1.set_title(f'{model_name} - Time Series', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, alpha=0.3)

    ax2 = plt.subplot(1, 3, 2)
    ax2.scatter(y_true, y_pred, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='1:1 Line')
    ax2.set_xlabel('Observed Discharge (cumecs)',  fontsize=12, fontweight='bold')
    ax2.set_ylabel('Predicted Discharge (cumecs)', fontsize=12, fontweight='bold')
    ax2.set_title(f'{model_name} - Scatter Plot',  fontsize=14, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)
    try:
        r2 = r2_score(y_true, y_pred)
        ax2.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax2.transAxes,
                 fontsize=11, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    except Exception:
        pass

    ax3 = plt.subplot(1, 3, 3)
    residuals = y_true - y_pred
    parts = ax3.violinplot([residuals], positions=[0], showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor('skyblue')
        pc.set_alpha(0.7)
    ax3.set_ylabel('Prediction Error (Residuals)', fontsize=12, fontweight='bold')
    ax3.set_title(f'{model_name} - Error Distribution', fontsize=14, fontweight='bold')
    ax3.set_xticks([0])
    ax3.set_xticklabels([model_name])
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.tight_layout()
    plt.show()


def plot_taylor_diagram(std_devs, correlations, model_names, ref_std):
    fig = plt.figure(figsize=(10, 10))
    ax  = fig.add_subplot(111, projection='polar')
    ax.plot(0, ref_std, 'k*', markersize=20, label='Observed', markeredgewidth=1.5)
    colors  = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'cyan', 'magenta']
    markers = ['o', 's', '^', 'D', 'v', 'P', '*', 'X']
    for i, (std, corr, name) in enumerate(zip(std_devs, correlations, model_names)):
        try:
            corr_val = float(corr)
            corr_val = np.clip(corr_val, -0.9999, 0.9999)
            angle = np.arccos(corr_val)
        except Exception:
            angle = 0.0
        ax.plot(angle, std, markers[i % len(markers)], markersize=12,
                color=colors[i % len(colors)], label=name,
                markeredgewidth=1.5, markeredgecolor='k')
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(1)
    ax.set_rlabel_position(0)
    ax.set_title('Taylor Diagram - Model Comparison', fontsize=16, fontweight='bold', pad=20)
    corr_labels = [0.0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0]
    ax.set_thetagrids(np.arccos(corr_labels) * 180 / np.pi, labels=corr_labels)
    ax.set_ylim(0, max(std_devs + [ref_std]) * 1.2)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    plt.tight_layout()
    plt.show()

# =====================================================================
# PART 10: RUN ALL ML MODELS FOR A SINGLE COM COMBINATION
# =====================================================================
def run_models_for_com(X_train, y_train, X_test, y_test, dates_test):
    """Trains all 7 ML models; returns results dict."""
    results = {}

    try:
        y_pred, _ = train_random_forest(X_train, y_train, X_test, y_test)
        results['Random Forest'] = {'predictions': y_pred,
                                    'metrics': calculate_metrics(y_test, y_pred, 'Random Forest')}
    except Exception as e:
        print(f"  ✗ Random Forest failed: {e}")

    try:
        y_pred, _ = train_svr(X_train, y_train, X_test, y_test,
                               MinMaxScaler(), MinMaxScaler())
        results['SVR (RBF)'] = {'predictions': y_pred,
                                'metrics': calculate_metrics(y_test, y_pred, 'SVR (RBF)')}
    except Exception as e:
        print(f"  ✗ SVR failed: {e}")

    try:
        y_pred, _ = train_linear_svr(X_train, y_train, X_test, y_test,
                                      MinMaxScaler(), MinMaxScaler())
        results['Linear SVM'] = {'predictions': y_pred,
                                 'metrics': calculate_metrics(y_test, y_pred, 'Linear SVM')}
    except Exception as e:
        print(f"  ✗ LinearSVR failed: {e}")

    try:
        y_pred, _ = train_lstm(X_train, y_train, X_test, y_test,
                                MinMaxScaler(), MinMaxScaler())
        results['LSTM'] = {'predictions': y_pred,
                           'metrics': calculate_metrics(y_test, y_pred, 'LSTM')}
    except Exception as e:
        print(f"  ✗ LSTM failed: {e}")

    try:
        y_pred, _ = train_gru(X_train, y_train, X_test, y_test,
                               MinMaxScaler(), MinMaxScaler())
        results['GRU'] = {'predictions': y_pred,
                          'metrics': calculate_metrics(y_test, y_pred, 'GRU')}
    except Exception as e:
        print(f"  ✗ GRU failed: {e}")

    try:
        y_pred, _ = train_cnn(X_train, y_train, X_test, y_test,
                               MinMaxScaler(), MinMaxScaler())
        results['CNN'] = {'predictions': y_pred,
                          'metrics': calculate_metrics(y_test, y_pred, 'CNN')}
    except Exception as e:
        print(f"  ✗ CNN failed: {e}")

    try:
        y_pred, _ = train_ann(X_train, y_train, X_test, y_test,
                               MinMaxScaler(), MinMaxScaler())
        results['ANN'] = {'predictions': y_pred,
                          'metrics': calculate_metrics(y_test, y_pred, 'ANN')}
    except Exception as e:
        print(f"  ✗ ANN failed: {e}")

    return results

# =====================================================================
# PART 11: BEST-COMBINATION SUMMARY TABLE (RMSE / MAE / R²)
# =====================================================================
def print_best_combination_table(master_df):
    """
    For each (Station, Model), finds:
      • Best COM by lowest  RMSE
      • Best COM by lowest  MAE
      • Best COM by highest R²
    Prints a single consolidated table.
    """
    print("\n" + "="*90)
    print("BEST FEATURE COMBINATION PER MODEL PER STATION  (RMSE ↓ | MAE ↓ | R² ↑)")
    print("="*90)

    # Filter to ML COMs only (exclude Rating Curve which appears as its own COM label)
    ml_df = master_df[master_df['COM'] != 'Rating Curve'].copy()

    if ml_df.empty:
        print("  No ML COM results available yet.")
        return

    rows = []
    for station in ml_df['Station'].unique():
        stn_df = ml_df[ml_df['Station'] == station]
        for model in stn_df['Model'].unique():
            mod_df = stn_df[stn_df['Model'] == model]
            if mod_df.empty:
                continue

            best_rmse_row = mod_df.loc[mod_df['RMSE'].idxmin()]
            best_mae_row  = mod_df.loc[mod_df['MAE'].idxmin()]
            best_r2_row   = mod_df.loc[mod_df['R²'].idxmax()]

            rows.append({
                'Station':         station,
                'Model':           model,
                'Best COM (RMSE)': best_rmse_row['COM'],
                'RMSE':            best_rmse_row['RMSE'],
                'Best COM (MAE)':  best_mae_row['COM'],
                'MAE':             best_mae_row['MAE'],
                'Best COM (R²)':   best_r2_row['COM'],
                'R²':              best_r2_row['R²'],
            })

    if not rows:
        print("  No data to display.")
        return

    best_df = pd.DataFrame(rows)
    col_order = ['Station', 'Model',
                 'Best COM (RMSE)', 'RMSE',
                 'Best COM (MAE)',  'MAE',
                 'Best COM (R²)',   'R²']
    best_df = best_df[col_order]
    print(best_df.to_string(index=False))
    print("="*90)
    print("  Interpretation: 'Best COM (RMSE)' = combination that yields the lowest RMSE")
    print("                  'Best COM (MAE)'  = combination that yields the lowest MAE")
    print("                  'Best COM (R²)'   = combination that yields the highest R²")
    print("="*90)

# =====================================================================
# PART 12: MAIN STATION PROCESSING WITH COM COMBINATIONS + RATING CURVE
# =====================================================================
def process_station(filepath, station_name, start_year_override=None):
    """
    Complete pipeline for one station:
      1. Load & preprocess data
      2. Rating Curve (statistical, COM1 / S_t only) with full visualisations
      3. All 7 COM combinations × all 7 ML models
      4. Per-COM metrics table + Taylor diagram
      5. Full station summary table
    Returns: (all_com_metrics list, summary DataFrame)
    """
    print("\n" + "="*70)
    print(f"{'PROCESSING STATION: ' + station_name:^70}")
    print("="*70)

    # ── Step 1: Preprocess ─────────────────────────────────────────────
    df = preprocess_data(filepath, start_year_override=start_year_override)

    if df is None or len(df) < 10:
        print(f"\n⚠ Insufficient data for {station_name}. Skipping...")
        return None, None

    # ── Step 2: Print COM table ────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"COM FEATURE COMBINATIONS — Station: {station_name}")
    print(f"{'='*70}")
    print(f"{'Combination':<12} {'No. Features':<15} {'Features'}")
    print("-"*70)
    for com_name, features in COM_FEATURES.items():
        print(f"{com_name:<12} {len(features):<15} {', '.join(features)}")
    print(f"{'Rating Curve':<12} {'1 (S_t)':<15} Statistical method — COM1 only")
    print("="*70)

    all_com_metrics = []  # accumulates dicts for all COMs and models

    # ── Step 3: RATING CURVE (statistical, uses S_t / COM1 only) ──────
    print(f"\n{'─'*70}")
    print(f"  Station: {station_name}  |  Rating Curve (Statistical)  |  Feature: S_t")
    print(f"{'─'*70}")

    # Build COM1 split for rating curve (stage only, no lag needed)
    rc_df = df.copy().dropna(subset=['Stage', 'Discharge']).reset_index(drop=True)
    dates_rc  = rc_df['Date'].values
    stage_all = rc_df['Stage'].values
    q_all     = rc_df['Discharge'].values

    years_rc  = pd.DatetimeIndex(dates_rc).year
    mask_test_rc = (years_rc >= 2015) & (years_rc <= 2020)
    min_test_rc  = max(5, int(0.05 * len(stage_all)))

    if mask_test_rc.sum() >= min_test_rc:
        stage_train_rc = stage_all[~mask_test_rc]
        q_train_rc     = q_all[~mask_test_rc]
        stage_test_rc  = stage_all[mask_test_rc]
        q_test_rc      = q_all[mask_test_rc]
        dates_test_rc  = dates_rc[mask_test_rc]
        print(f"  Test period 2015–2020: {mask_test_rc.sum()} samples | "
              f"Train: {(~mask_test_rc).sum()} samples")
    else:
        split_idx      = int(len(stage_all) * 0.8)
        stage_train_rc = stage_all[:split_idx]
        q_train_rc     = q_all[:split_idx]
        stage_test_rc  = stage_all[split_idx:]
        q_test_rc      = q_all[split_idx:]
        dates_test_rc  = dates_rc[split_idx:]
        print(f"  ⚠ Fallback 80/20 split | Train: {split_idx} | Test: {len(stage_all)-split_idx}")

    try:
        y_pred_rc, rc_params = train_rating_curve(stage_train_rc, q_train_rc, stage_test_rc)
        rc_metrics = calculate_metrics(q_test_rc, y_pred_rc, 'Rating Curve')

        print(f"\n  ── Rating Curve Metrics ({station_name}) ──")
        rc_metrics_df = pd.DataFrame([rc_metrics])
        print(rc_metrics_df.to_string(index=False))

        # Four-panel visualisation
        visualize_rating_curve(
            stage_train_rc, q_train_rc,
            stage_test_rc,  q_test_rc,
            y_pred_rc, rc_params,
            f"{station_name} | Rating Curve",
            dates_test_rc
        )

        # Add to collector (labelled as 'Rating Curve' COM for the summary)
        rc_row = rc_metrics.copy()
        rc_row['Station']  = station_name
        rc_row['COM']      = 'Rating Curve'
        rc_row['Features'] = 'S_t (statistical)'
        all_com_metrics.append(rc_row)

    except Exception as e:
        print(f"  ✗ Rating Curve failed: {e}")

    # ── Step 4: ML models — all 7 COM combinations ────────────────────
    for com_name, feature_list in COM_FEATURES.items():

        print(f"\n{'─'*70}")
        print(f"  Station: {station_name}  |  {com_name}  |  Features: {', '.join(feature_list)}")
        print(f"{'─'*70}")

        X, y, dates = create_com_features(df, com_name, feature_list)

        if len(X) < 10:
            print(f"  ⚠ Not enough samples for {com_name} ({len(X)} rows). Skipping.")
            continue

        years    = pd.DatetimeIndex(dates).year
        mask_test = (years >= 2015) & (years <= 2020)
        min_test  = max(5, int(0.05 * len(X)))

        if mask_test.sum() >= min_test:
            X_train, X_test = X[~mask_test], X[mask_test]
            y_train, y_test = y[~mask_test], y[mask_test]
            dates_test      = dates[mask_test]
            print(f"  Test period 2015–2020: {mask_test.sum()} samples | "
                  f"Train: {(~mask_test).sum()} samples")
        else:
            split_idx       = int(len(X) * 0.8)
            X_train, X_test = X[:split_idx], X[split_idx:]
            y_train, y_test = y[:split_idx], y[split_idx:]
            dates_test      = dates[split_idx:]
            print(f"  ⚠ Fallback 80/20 split | Train: {len(X_train)} | Test: {len(X_test)}")

        results = run_models_for_com(X_train, y_train, X_test, y_test, dates_test)

        if not results:
            print(f"  ⚠ No successful models for {com_name}.")
            continue

        # Per-COM metrics table
        print(f"\n  ── {com_name} Metrics ({station_name}) ──")
        com_metrics_df = pd.DataFrame([results[m]['metrics'] for m in results])
        print(com_metrics_df.to_string(index=False))

        # Visualisations
        for model_name in results:
            y_pred = results[model_name]['predictions']
            visualize_results(y_test, y_pred,
                              f"{station_name} | {com_name} | {model_name}",
                              dates_test)

        # Taylor Diagram
        std_devs       = [np.std(results[m]['predictions']) for m in results]
        correlations   = [results[m]['metrics']['Correlation'] for m in results]
        model_names_td = list(results.keys())
        ref_std        = np.std(y_test)
        print(f"\n  Creating Taylor Diagram for {com_name}...")
        plot_taylor_diagram(std_devs, correlations, model_names_td, ref_std)

        # Collect metrics
        for model_name, res in results.items():
            m            = res['metrics'].copy()
            m['Station'] = station_name
            m['COM']     = com_name
            m['Features']= ', '.join(feature_list)
            all_com_metrics.append(m)

    # ── Step 5: Full station summary table ────────────────────────────
    if all_com_metrics:
        summary_df = pd.DataFrame(all_com_metrics)
        col_order  = ['Station', 'COM', 'Features', 'Model', 'RMSE', 'MAE', 'R²', 'NSE', 'Correlation']
        summary_df = summary_df[[c for c in col_order if c in summary_df.columns]]

        print(f"\n{'='*70}")
        print(f"  FULL SUMMARY — Station: {station_name} — Rating Curve + ALL COM × ALL MODELS")
        print(f"{'='*70}")
        print(summary_df.to_string(index=False))
        print(f"{'='*70}")
    else:
        summary_df = pd.DataFrame()

    print(f"\n{'='*70}")
    print(f"✓ COMPLETED: {station_name}")
    print(f"{'='*70}")

    return all_com_metrics, summary_df

# =====================================================================
# PART 13: EXECUTE FOR ALL STATIONS
# =====================================================================

# Print COM table once at start
print_com_features_table()

# Define file paths — UPDATE PATHS IN GOOGLE COLAB IF NEEDED
files = {
    'Bamnidhi':   ('Discharge data Bamnidhi 2020.xlsx',   None),
    'Sundergarh': ('Discharge data Sundergarh 2020.xlsx', None),
    'Kesinga':    ('Discharge data Kesinga 2020.xlsx',    None)
}

all_station_results = {}

for station_name, (filepath, override_year) in files.items():
    try:
        com_metrics, summary_df = process_station(
            filepath, station_name, start_year_override=override_year
        )
        all_station_results[station_name] = {
            'com_metrics': com_metrics,
            'summary_df':  summary_df
        }
    except Exception as e:
        print(f"\n✗ Error processing {station_name}: {e}")
        continue

# =====================================================================
# PART 14: CROSS-STATION COMPARATIVE ANALYSIS
# =====================================================================
print("\n" + "="*70)
print("CROSS-STATION COMPARATIVE ANALYSIS — Rating Curve + ALL COMs × ALL MODELS")
print("="*70)

all_frames = []
for stn, data in all_station_results.items():
    df_stn = data.get('summary_df')
    if df_stn is not None and not df_stn.empty:
        all_frames.append(df_stn)

if all_frames:
    master_df = pd.concat(all_frames, ignore_index=True)
    print(master_df.to_string(index=False))

    # ── Best COM per Station × Model (RMSE / MAE / R²) ────────────────
    print_best_combination_table(master_df)

    # ── Best overall (NSE) — existing behaviour kept ───────────────────
    if 'NSE' in master_df.columns:
        ml_master = master_df[master_df['COM'] != 'Rating Curve']
        if not ml_master.empty:
            print("\n" + "="*70)
            print("BEST MODEL PER STATION × COM (by highest NSE)")
            print("="*70)
            best = ml_master.loc[ml_master.groupby(['Station', 'COM'])['NSE'].idxmax()]
            best_display = best[['Station', 'COM', 'Model', 'NSE', 'R²', 'RMSE']].reset_index(drop=True)
            print(best_display.to_string(index=False))
else:
    print("No results available for comparison.")

print("\n" + "="*70)
print("✓ ALL PROCESSING COMPLETE!")
print("="*70)

In [ ]:
# =====================================================================
# STANDALONE CELL — AUGUST 2017 DISCHARGE FORECAST
# Run this cell AFTER the main script cell (Parts 1–11 must be executed).
# Predicts daily discharge for ALL 31 days of August 2017:
#   • Rating Curve   (COM1 / S_t only)
#   • 7 ML models    × 7 COM feature combinations
# for all three stations: Bamnidhi, Sundergarh, Kesinga
# Output: rich console tables + CSV/Excel export
# =====================================================================

import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler

# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION — update paths to match your Colab upload location
# ─────────────────────────────────────────────────────────────────────
STATION_FILES = {
    'Bamnidhi':   'Discharge data Bamnidhi 2020.xlsx',
    'Sundergarh': 'Discharge data Sundergarh 2020.xlsx',
    'Kesinga':    'Discharge data Kesinga 2020.xlsx',
}

# Target month for prediction
PRED_YEAR  = 2017
PRED_MONTH = 8   # August

# ─────────────────────────────────────────────────────────────────────
# HELPER — Build a clean August 2017 test window WITH seed rows
# ─────────────────────────────────────────────────────────────────────
def get_aug2017_split(df, com_name, feature_list, seed_days=3):
    """
    Returns (X_train, y_train, X_test, y_test, dates_test) where:
      • train  = all data strictly before August 2017
      • test   = all rows whose Date falls in August 2017
    Lag features are seeded correctly from the days just before Aug 1.

    Parameters
    ----------
    df           : full preprocessed DataFrame (Date, Stage, Discharge)
    com_name     : e.g. 'COM1'
    feature_list : list of feature column names for that COM
    seed_days    : how many days before Aug 1 we need for lag seed
                   (max lag in COM7 is 3 days, so default=3 is correct)
    """
    df = df.copy().sort_values('Date').reset_index(drop=True)

    # Build ALL lag columns over the full series so lags are correct
    df['S_t']    = df['Stage']
    df['S_lag1'] = df['Stage'].shift(1)
    df['S_lag2'] = df['Stage'].shift(2)
    df['S_lag3'] = df['Stage'].shift(3)
    df['Q_lag1'] = df['Discharge'].shift(1)
    df['Q_lag2'] = df['Discharge'].shift(2)
    df['Q_lag3'] = df['Discharge'].shift(3)

    aug_start = datetime(PRED_YEAR, PRED_MONTH, 1)
    aug_end   = datetime(PRED_YEAR, PRED_MONTH, 31)

    needed_cols = feature_list + ['Discharge', 'Date']
    df_feat = df[needed_cols].dropna().reset_index(drop=True)

    mask_aug   = (df_feat['Date'] >= aug_start) & (df_feat['Date'] <= aug_end)
    mask_train = df_feat['Date'] < aug_start

    if mask_aug.sum() == 0:
        return None, None, None, None, None

    X_train    = df_feat.loc[mask_train, feature_list].values
    y_train    = df_feat.loc[mask_train, 'Discharge'].values
    X_test     = df_feat.loc[mask_aug,   feature_list].values
    y_test     = df_feat.loc[mask_aug,   'Discharge'].values
    dates_test = df_feat.loc[mask_aug,   'Date'].values

    if len(X_train) < 5:
        return None, None, None, None, None

    return X_train, y_train, X_test, y_test, dates_test


# ─────────────────────────────────────────────────────────────────────
# HELPER — Rating Curve for August 2017
# ─────────────────────────────────────────────────────────────────────
def get_rating_curve_aug2017(df):
    """
    Train RC on all data before August 2017,
    predict for the 31 days of August 2017.
    Returns (dates_test, y_test_obs, y_pred_rc) or (None,None,None).
    """
    rc_df = df.copy().dropna(subset=['Stage', 'Discharge']).sort_values('Date').reset_index(drop=True)

    aug_start = datetime(PRED_YEAR, PRED_MONTH, 1)
    aug_end   = datetime(PRED_YEAR, PRED_MONTH, 31)

    mask_aug   = (rc_df['Date'] >= aug_start) & (rc_df['Date'] <= aug_end)
    mask_train = rc_df['Date'] < aug_start

    if mask_aug.sum() == 0:
        return None, None, None

    stage_train = rc_df.loc[mask_train, 'Stage'].values
    q_train     = rc_df.loc[mask_train, 'Discharge'].values
    stage_test  = rc_df.loc[mask_aug,   'Stage'].values
    q_test      = rc_df.loc[mask_aug,   'Discharge'].values
    dates_test  = rc_df.loc[mask_aug,   'Date'].values

    try:
        y_pred_rc, rc_params = train_rating_curve(stage_train, q_train, stage_test)
        return dates_test, q_test, y_pred_rc
    except Exception as e:
        print(f"    ✗ Rating Curve failed: {e}")
        return None, None, None


# ─────────────────────────────────────────────────────────────────────
# HELPER — Train one ML model and predict for August 2017
# ─────────────────────────────────────────────────────────────────────
def train_and_predict_ml(model_name, X_train, y_train, X_test, y_test):
    """
    Dispatches to the appropriate training function from the main script.
    Returns y_pred (numpy array) or None on failure.
    """
    sx, sy = MinMaxScaler(), MinMaxScaler()
    try:
        if model_name == 'Random Forest':
            y_pred, _ = train_random_forest(X_train, y_train, X_test, y_test)
        elif model_name == 'SVR (RBF)':
            y_pred, _ = train_svr(X_train, y_train, X_test, y_test, sx, sy)
        elif model_name == 'Linear SVM':
            y_pred, _ = train_linear_svr(X_train, y_train, X_test, y_test, sx, sy)
        elif model_name == 'LSTM':
            y_pred, _ = train_lstm(X_train, y_train, X_test, y_test, sx, sy)
        elif model_name == 'GRU':
            y_pred, _ = train_gru(X_train, y_train, X_test, y_test, sx, sy)
        elif model_name == 'CNN':
            y_pred, _ = train_cnn(X_train, y_train, X_test, y_test, sx, sy)
        elif model_name == 'ANN':
            y_pred, _ = train_ann(X_train, y_train, X_test, y_test, sx, sy)
        else:
            return None
        return y_pred
    except Exception as e:
        print(f"      ✗ {model_name} failed: {e}")
        return None


# ─────────────────────────────────────────────────────────────────────
# MAIN LOOP — Iterate stations → models → COM combinations
# ─────────────────────────────────────────────────────────────────────
ML_MODELS = ['Random Forest', 'SVR (RBF)', 'Linear SVM', 'LSTM', 'GRU', 'CNN', 'ANN']

# Master collector:  each row = one (station, com, model, date) prediction
all_records = []

print("\n" + "="*80)
print("  AUGUST 2017 DISCHARGE FORECAST — ALL STATIONS × ALL MODELS × ALL COMs")
print("="*80)

for station_name, filepath in STATION_FILES.items():

    print(f"\n{'─'*80}")
    print(f"  STATION: {station_name}   |   file: {filepath}")
    print(f"{'─'*80}")

    # ── Load & preprocess data ────────────────────────────────────────
    try:
        df_full = preprocess_data(filepath)
    except Exception as e:
        print(f"  ✗ Could not load {station_name}: {e}")
        continue

    if df_full is None or len(df_full) < 20:
        print(f"  ✗ Insufficient data for {station_name}. Skipping.")
        continue

    # Verify August 2017 exists in the dataset
    aug_check = df_full[
        (df_full['Date'].dt.year  == PRED_YEAR) &
        (df_full['Date'].dt.month == PRED_MONTH)
    ]
    if len(aug_check) == 0:
        print(f"  ✗ No August {PRED_YEAR} data found in {station_name}. Skipping.")
        continue
    print(f"  ✓ August {PRED_YEAR} observed rows available: {len(aug_check)}")

    # ── Rating Curve ──────────────────────────────────────────────────
    print(f"\n  [RC] Rating Curve (COM1 / S_t only) ...")
    dates_rc, q_obs_rc, q_pred_rc = get_rating_curve_aug2017(df_full)

    if dates_rc is not None:
        rc_metrics = calculate_metrics(q_obs_rc, q_pred_rc, 'Rating Curve')
        print(f"       RMSE={rc_metrics['RMSE']:.4f}  MAE={rc_metrics['MAE']:.4f}  "
              f"R²={rc_metrics['R²']:.4f}  NSE={rc_metrics['NSE']:.4f}")
        for d, obs, pred in zip(dates_rc, q_obs_rc, q_pred_rc):
            all_records.append({
                'Station':    station_name,
                'COM':        'COM1',
                'Features':   'S_t (statistical)',
                'Model':      'Rating Curve',
                'Date':       pd.Timestamp(d),
                'Day':        pd.Timestamp(d).day,
                'Observed':   round(float(obs),  4),
                'Predicted':  round(float(pred), 4),
                'RMSE':       rc_metrics['RMSE'],
                'MAE':        rc_metrics['MAE'],
                'R²':         rc_metrics['R²'],
                'NSE':        rc_metrics['NSE'],
            })

    # ── ML models × all COM combinations ─────────────────────────────
    for com_name, feature_list in COM_FEATURES.items():

        print(f"\n  [{com_name}] Features: {', '.join(feature_list)}")

        result = get_aug2017_split(df_full, com_name, feature_list)
        X_train, y_train, X_test, y_test, dates_test = result

        if X_train is None:
            print(f"    ✗ Could not build feature matrix for {com_name}. Skipping.")
            continue

        print(f"    Train samples: {len(X_train)} | Aug-2017 test samples: {len(X_test)}")

        for model_name in ML_MODELS:
            print(f"    Training {model_name} ...", end=' ', flush=True)
            y_pred = train_and_predict_ml(model_name, X_train, y_train, X_test, y_test)

            if y_pred is None:
                continue

            metrics = calculate_metrics(y_test, y_pred, model_name)
            print(f"RMSE={metrics['RMSE']:.4f}  R²={metrics['R²']:.4f}")

            for d, obs, pred in zip(dates_test, y_test, y_pred):
                all_records.append({
                    'Station':   station_name,
                    'COM':       com_name,
                    'Features':  ', '.join(feature_list),
                    'Model':     model_name,
                    'Date':      pd.Timestamp(d),
                    'Day':       pd.Timestamp(d).day,
                    'Observed':  round(float(obs),  4),
                    'Predicted': round(float(pred), 4),
                    'RMSE':      metrics['RMSE'],
                    'MAE':       metrics['MAE'],
                    'R²':        metrics['R²'],
                    'NSE':       metrics['NSE'],
                })

# ─────────────────────────────────────────────────────────────────────
# BUILD RESULTS DataFrame
# ─────────────────────────────────────────────────────────────────────
if not all_records:
    print("\n⚠  No results collected. Check that all files are loaded correctly.")
else:
    results_df = pd.DataFrame(all_records)
    results_df = results_df.sort_values(
        ['Station', 'COM', 'Model', 'Date']
    ).reset_index(drop=True)

    # ── 1. LONG TABLE — every prediction ─────────────────────────────
    print("\n\n" + "="*80)
    print("  LONG RESULTS TABLE  (Station | COM | Model | Date | Observed | Predicted)")
    print("="*80)
    display_cols = ['Station', 'COM', 'Model', 'Date', 'Day',
                    'Observed', 'Predicted', 'RMSE', 'MAE', 'R²', 'NSE']
    print(results_df[display_cols].to_string(index=False))

    # ── 2. DAILY PIVOT — wide format (Day 1…31 as columns) ───────────
    print("\n\n" + "="*80)
    print("  DAILY PIVOT TABLE  (rows = Station|COM|Model, columns = Day 1–31)")
    print("="*80)
    pivot = results_df.pivot_table(
        index=['Station', 'COM', 'Model'],
        columns='Day',
        values='Predicted',
        aggfunc='first'
    )
    pivot.columns = [f"Aug-{int(c):02d}" for c in pivot.columns]
    print(pivot.to_string())

    # ── 3. METRICS SUMMARY — one row per (Station, COM, Model) ───────
    print("\n\n" + "="*80)
    print("  METRICS SUMMARY FOR AUGUST 2017  (RMSE | MAE | R² | NSE)")
    print("="*80)
    metrics_summary = (
        results_df
        .groupby(['Station', 'COM', 'Features', 'Model'], sort=False)[['RMSE','MAE','R²','NSE']]
        .first()
        .reset_index()
    )
    print(metrics_summary.to_string(index=False))

    # ── 4. BEST MODEL PER STATION (lowest RMSE in August 2017) ───────
    print("\n\n" + "="*80)
    print("  BEST MODEL × COM PER STATION — August 2017  (by lowest RMSE)")
    print("="*80)
    best_per_station = (
        metrics_summary
        .loc[metrics_summary.groupby('Station')['RMSE'].idxmin()]
        [['Station', 'COM', 'Model', 'RMSE', 'MAE', 'R²', 'NSE']]
        .reset_index(drop=True)
    )
    print(best_per_station.to_string(index=False))

    # ── 5. EXPORT ─────────────────────────────────────────────────────
    out_long    = f"Aug{PRED_YEAR}_predictions_long.csv"
    out_pivot   = f"Aug{PRED_YEAR}_predictions_pivot.csv"
    out_metrics = f"Aug{PRED_YEAR}_metrics_summary.csv"
    out_excel   = f"Aug{PRED_YEAR}_discharge_forecast.xlsx"

    results_df[display_cols].to_csv(out_long,    index=False)
    pivot.to_csv(out_pivot)
    metrics_summary.to_csv(out_metrics, index=False)

    # Multi-sheet Excel
    try:
        with pd.ExcelWriter(out_excel, engine='openpyxl') as writer:
            results_df[display_cols].to_excel(
                writer, sheet_name='Daily_Predictions', index=False)
            pivot.to_excel(
                writer, sheet_name='Pivot_Day1_to_31')
            metrics_summary.to_excel(
                writer, sheet_name='Metrics_Summary', index=False)
            best_per_station.to_excel(
                writer, sheet_name='Best_Model_per_Station', index=False)
        print(f"\n✓ Excel workbook saved  →  {out_excel}")
    except Exception as e:
        print(f"\n⚠ Excel export failed ({e}). CSVs still saved.")

    print(f"✓ Long CSV saved        →  {out_long}")
    print(f"✓ Pivot CSV saved       →  {out_pivot}")
    print(f"✓ Metrics CSV saved     →  {out_metrics}")

    print("\n" + "="*80)
    print("  ✓  AUGUST 2017 FORECAST COMPLETE")
    print(f"     Total prediction rows : {len(results_df)}")
    print(f"     Stations processed    : {results_df['Station'].nunique()}")
    print(f"     COM combinations      : {results_df['COM'].nunique()}")
    print(f"     Unique models         : {results_df['Model'].nunique()}")
    print("="*80)